# Train U-Net

Original ver2 model and training settings. Install the package first; configure `SEAICE_CONFIG` before starting the kernel. This notebook trains only. Use the package CLI for ocean-masked manuscript evaluation.


## Export-only execution after kernel restart

To create the manuscript regional-zoom field archives without retraining:

1. Run the setup/import/cache/dataset cells before the training cell.
2. Keep `UNET_FORCE_RETRAIN = False` and `UNET_LOAD_IF_EXISTS = True`.
3. Run the model/checkpoint cell. It should print `Loading U-Net checkpoint`.
4. Run the metric/function-definition cell that defines `full_predict_daily`.
5. Skip cells that regenerate all test metrics or all Cartopy monthly figures unless needed.
6. Run `## Export selected forecast fields for manuscript regional zoom`.

The export cell only forwards the two selected 2025 initialization cases and writes `.npz` field archives.




In [ ]:
from seaice_diagnostics.config import CACHE, ERA5, TRAINING, raw_path
# ============================================================
# IMPORTS
# ============================================================

import glob
import math
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

import torch
import torch.nn as nn
from torch.utils.data import Dataset, Subset, DataLoader

import matplotlib.pyplot as plt
from tqdm.auto import tqdm


torch.backends.cudnn.benchmark = True



In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

@dataclass
class DataPaths:
    era5_glob: str = raw_path("era5_glob")
    era5_sample: str = str(ERA5)
    grid_daily_cache_dir: str = str(CACHE)


EXPERIMENT_NAME = "seaice_prediction"
BASELINE_TAG = "simple_unet_epoch20_patience3_gridcache_1979_2025_ver2"

HISTORY_LEN = 15
HORIZON_DAYS = 30
USED_VARIABLES = ["t2m", "d2m", "u10", "v10", "msl", "sst", "sic", "sit"]
NUM_VARS = len(USED_VARIABLES)

EXPERIMENT_START_DATE = "1979-01-01"
EXPERIMENT_END_DATE = "2025-12-31"

TRAIN_START_DATE = "1979-01-01"
TRAIN_END_DATE = "2019-12-31"
VAL_START_DATE = "2020-01-01"
VAL_END_DATE = "2022-12-31"
TEST_START_DATE = "2023-01-01"
TEST_END_DATE = "2025-12-31"
TEST_YEARS = [2023, 2024, 2025]
FIGURE_YEARS = [2025]

ICE_EDGE_THRESHOLD = 0.15

# U-Net training settings. Full daily training is retained by default for fair
# period consistency. Increase TRAIN_INDEX_STEP only for quick debugging.
UNET_EPOCHS = 20
UNET_PATIENCE = 3
UNET_BATCH_SIZE = 1
UNET_BASE_CHANNELS = 16
UNET_LR = 1e-3
UNET_WEIGHT_DECAY = 1e-5
TRAIN_INDEX_STEP = 1
VAL_INDEX_STEP = 1

# Statistics are computed from the training period only. Set to 1 for exact
# daily statistics; 7 is faster and usually sufficient for a baseline test run.
CHANNEL_STATS_SAMPLE_STEP = 7
FORCE_REBUILD_CHANNEL_STATS = False

UNET_FORCE_RETRAIN = False
UNET_LOAD_IF_EXISTS = True

paths = DataPaths()
OUTPUT_DIR = (TRAINING / "unet")
FIGURE_DIR = (TRAINING / "unet" / "figures")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
for p in [OUTPUT_DIR, FIGURE_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
for year in TEST_YEARS:
    (OUTPUT_DIR / f"test_{year}").mkdir(parents=True, exist_ok=True)
    (FIGURE_DIR / f"test_{year}").mkdir(parents=True, exist_ok=True)

channel_stats_path = OUTPUT_DIR / "channel_statistics.npz"
unet_checkpoint_path = CHECKPOINT_DIR / "best.pth"
training_history_path = OUTPUT_DIR / "training_history_simple_unet_epoch20_patience3_1979_2025_ver2.csv"

print("output:", OUTPUT_DIR)
print("figures:", FIGURE_DIR)
print("checkpoint:", unet_checkpoint_path)



In [ ]:
# ============================================================
# DATE, GRID, AND DOMAIN SETUP
# ============================================================

def _cache_date_key(date_like) -> str:
    return pd.Timestamp(date_like).strftime("%Y%m%d")


era5_file_list = sorted(glob.glob(paths.era5_glob))
if not era5_file_list:
    raise FileNotFoundError(f"No ERA5 files found: {paths.era5_glob}")
print("ERA5 file count:", len(era5_file_list))

cache_dir = Path(paths.grid_daily_cache_dir)
if not cache_dir.exists():
    raise FileNotFoundError(f"Grid cache directory not found: {cache_dir}")

all_era5 = xr.open_mfdataset(era5_file_list, combine="by_coords", engine="netcdf4", parallel=False)
all_dates = pd.DatetimeIndex(all_era5.time.values).normalize()
date_mask = (
    (all_dates >= pd.Timestamp(EXPERIMENT_START_DATE))
    & (all_dates <= pd.Timestamp(EXPERIMENT_END_DATE))
)
valid_dates = pd.DatetimeIndex(all_dates[date_mask])
lat_values = np.asarray(all_era5.latitude.values)
lon_values = np.asarray(all_era5.longitude.values)
num_lat = len(lat_values)
num_lon = len(lon_values)
all_era5.close()

expected_dates = pd.date_range(EXPERIMENT_START_DATE, EXPERIMENT_END_DATE, freq="D")
if not valid_dates.equals(expected_dates):
    missing = expected_dates.difference(valid_dates)
    raise ValueError(
        f"ERA5 daily coverage is incomplete for {EXPERIMENT_START_DATE} to {EXPERIMENT_END_DATE}. "
        f"Missing examples: {missing[:5].tolist()}"
    )

missing_cache = [d for d in valid_dates if not (cache_dir / f"{_cache_date_key(d)}.npz").exists()]
if missing_cache:
    raise FileNotFoundError(f"Missing grid cache examples: {missing_cache[:5]}")
print("valid_dates:", len(valid_dates), valid_dates.min(), "~", valid_dates.max())
print("grid:", num_lat, "x", num_lon)


def _lon_to_360(lon):
    return np.mod(np.asarray(lon, dtype=np.float32), 360.0)


def _lon_between_360(lon360, lon_min, lon_max):
    lon_min = float(lon_min) % 360.0
    lon_max = float(lon_max) % 360.0
    if lon_min <= lon_max:
        return (lon360 >= lon_min) & (lon360 <= lon_max)
    return (lon360 >= lon_min) | (lon360 <= lon_max)


def get_grid_coordinates_from_era5(sample_path):
    ds = xr.open_dataset(sample_path)
    lat = np.asarray(ds.latitude.values)
    lon = np.asarray(ds.longitude.values)
    ds.close()
    lon_grid, lat_grid = np.meshgrid(lon, lat)
    return np.column_stack([lat_grid.ravel(), lon_grid.ravel()]).astype(np.float32)


NSR_REGION_BOXES = [
    ("chukchi", 66.0, 78.0, 180.0, 205.0),
    ("east_siberian", 68.0, 82.5, 140.0, 180.0),
    ("laptev", 70.0, 82.5, 100.0, 140.0),
    ("kara", 68.0, 82.5, 60.0, 100.0),
    ("barents", 68.0, 82.5, 20.0, 60.0),
]
NSR_REGION_NAMES = [box[0] for box in NSR_REGION_BOXES]


def build_nsr_region_masks(grid_coords, num_lat, num_lon, region_boxes):
    lat = np.asarray(grid_coords[:, 0], dtype=np.float32)
    lon360 = _lon_to_360(grid_coords[:, 1])
    masks = {}
    for name, lat_min, lat_max, lon_min, lon_max in region_boxes:
        in_region = (
            (lat >= float(lat_min))
            & (lat <= float(lat_max))
            & _lon_between_360(lon360, lon_min, lon_max)
        )
        masks[name] = in_region.reshape(num_lat, num_lon)
        print(f"NSR region {name:14s}: {int(in_region.sum()):6d} / {int(lat.shape[0])}")
    masks["nsr_corridor"] = np.logical_or.reduce([masks[name] for name in NSR_REGION_NAMES])
    print("NSR corridor union :", int(masks["nsr_corridor"].sum()), "/", int(lat.shape[0]))
    return masks


grid_coords = get_grid_coordinates_from_era5(paths.era5_sample)
nsr_region_masks_2d = build_nsr_region_masks(grid_coords, num_lat, num_lon, NSR_REGION_BOXES)
analysis_domain_masks_2d = {
    "pan_arctic": np.ones((num_lat, num_lon), dtype=bool),
    "nsr_corridor": nsr_region_masks_2d["nsr_corridor"],
}
analysis_domain_masks_2d.update({name: nsr_region_masks_2d[name] for name in NSR_REGION_NAMES})
ANALYSIS_DOMAINS = list(analysis_domain_masks_2d.keys())
print("analysis domains:", ANALYSIS_DOMAINS)



In [ ]:
# ============================================================
# DATASET, SPLIT, AND TRAINING-PERIOD NORMALIZATION
# ============================================================

class GridCacheDirectDataset(Dataset):
    def __init__(
        self,
        cache_dir,
        valid_dates,
        variables,
        history_len=15,
        horizon=30,
        channel_mean=None,
        channel_std=None,
        recent_cache_size=96,
    ):
        self.cache_dir = Path(cache_dir)
        self.valid_dates = pd.DatetimeIndex(valid_dates)
        self.variables = list(variables)
        self.history_len = int(history_len)
        self.horizon = int(horizon)
        self.num_vars = len(self.variables)
        self.recent_cache_size = int(recent_cache_size)
        self._recent = {}

        with np.load(self.cache_dir / f"{_cache_date_key(self.valid_dates[0])}.npz") as sample:
            self.num_lat = int(sample["daily"].shape[1])
            self.num_lon = int(sample["daily"].shape[2])
        self.input_channels = self.history_len * self.num_vars

        self.channel_mean = None if channel_mean is None else np.asarray(channel_mean, dtype=np.float32)
        self.channel_std = None if channel_std is None else np.asarray(channel_std, dtype=np.float32)
        if self.channel_mean is not None and self.channel_mean.shape[0] != self.num_vars:
            raise ValueError("channel_mean length must match variables.")
        if self.channel_std is not None and self.channel_std.shape[0] != self.num_vars:
            raise ValueError("channel_std length must match variables.")

    def __len__(self):
        return len(self.valid_dates) - self.history_len - self.horizon + 1

    def _load_day(self, date_like):
        key = _cache_date_key(date_like)
        if key in self._recent:
            return self._recent[key]
        path = self.cache_dir / f"{key}.npz"
        if not path.exists():
            raise FileNotFoundError(f"cache file missing: {path}")
        with np.load(path) as arr:
            day = {
                "daily": arr["daily"].astype(np.float32),
                "sic_full": arr["sic_full"].astype(np.float32),
            }
        self._recent[key] = day
        if len(self._recent) > self.recent_cache_size:
            self._recent.pop(next(iter(self._recent)))
        return day

    def current_pos(self, idx):
        return int(idx) + self.history_len - 1

    def target_positions(self, idx):
        curr = self.current_pos(idx)
        return np.arange(curr + 1, curr + 1 + self.horizon, dtype=int)

    def current_date(self, idx):
        return pd.Timestamp(self.valid_dates[self.current_pos(idx)])

    def target_dates(self, idx):
        return pd.DatetimeIndex(self.valid_dates[self.target_positions(idx)])

    def __getitem__(self, idx):
        idx = int(idx)
        hist_positions = np.arange(idx, idx + self.history_len, dtype=int)
        target_positions = self.target_positions(idx)

        hist_days = [self._load_day(self.valid_dates[pos]) for pos in hist_positions]
        target_days = [self._load_day(self.valid_dates[pos]) for pos in target_positions]

        daily_hist = np.stack([day["daily"] for day in hist_days], axis=0)
        full_hist_sic = np.stack([day["sic_full"] for day in hist_days], axis=0)
        full_target = np.stack([day["sic_full"] for day in target_days], axis=0)
        curr_sic = full_hist_sic[-1]

        if self.channel_mean is not None and self.channel_std is not None:
            daily_hist = (daily_hist - self.channel_mean[None, :, None, None]) / self.channel_std[None, :, None, None]

        x = daily_hist.reshape(self.history_len * self.num_vars, self.num_lat, self.num_lon)
        return {
            "x": torch.tensor(np.nan_to_num(x, nan=0.0), dtype=torch.float32),
            "curr_sic": torch.tensor(curr_sic[None, :, :], dtype=torch.float32),
            "y_full": torch.tensor(full_target, dtype=torch.float32),
            "start_idx": torch.tensor(idx, dtype=torch.long),
        }


def get_direct_target_date_frame(dataset):
    rows = []
    for idx in range(len(dataset)):
        targets = dataset.target_dates(idx)
        rows.append(
            {
                "sample_idx": idx,
                "init_date": dataset.current_date(idx).normalize(),
                "target_start": targets[0].normalize(),
                "target_end": targets[-1].normalize(),
            }
        )
    return pd.DataFrame(rows)


def split_direct_dataset_by_complete_target_window(dataset):
    frame = get_direct_target_date_frame(dataset)
    split_windows = {
        "train": (TRAIN_START_DATE, TRAIN_END_DATE),
        "val": (VAL_START_DATE, VAL_END_DATE),
        "test": (TEST_START_DATE, TEST_END_DATE),
    }
    result = {}
    print("===== complete-target-window train/val/test split =====")
    for name, (start, end) in split_windows.items():
        start = pd.Timestamp(start).normalize()
        end = pd.Timestamp(end).normalize()
        idx = frame.loc[
            (frame["target_start"] >= start) & (frame["target_end"] <= end),
            "sample_idx",
        ].astype(int).tolist()
        if not idx:
            raise ValueError(f"Empty {name} split detected.")
        result[name] = Subset(dataset, idx)
        sub = frame.loc[frame["sample_idx"].isin(idx)]
        print(
            f"{name:5s}: {len(idx):5d} samples | "
            f"target window {sub.target_start.min().date()} ~ {sub.target_end.max().date()}"
        )
    return result["train"], result["val"], result["test"], frame


def build_or_load_channel_stats(cache_dir, valid_dates, train_start, train_end, sample_step=7, force=False):
    if channel_stats_path.exists() and not force:
        stats = np.load(channel_stats_path)
        print("Loading channel stats:", channel_stats_path)
        return stats["mean"].astype(np.float32), stats["std"].astype(np.float32)

    dates = pd.date_range(train_start, train_end, freq="D")[::max(1, int(sample_step))]
    sums = np.zeros(NUM_VARS, dtype=np.float64)
    sums_sq = np.zeros(NUM_VARS, dtype=np.float64)
    counts = np.zeros(NUM_VARS, dtype=np.float64)

    for date in tqdm(dates, desc="training-period channel stats"):
        with np.load(Path(cache_dir) / f"{_cache_date_key(date)}.npz") as npz:
            arr = npz["daily"].astype(np.float64)
        for v in range(NUM_VARS):
            vals = arr[v]
            valid = np.isfinite(vals)
            if valid.any():
                x = vals[valid]
                sums[v] += x.sum()
                sums_sq[v] += np.square(x).sum()
                counts[v] += valid.sum()

    mean = sums / np.maximum(counts, 1)
    var = sums_sq / np.maximum(counts, 1) - np.square(mean)
    std = np.sqrt(np.maximum(var, 1e-12))
    std = np.where(std < 1e-6, 1.0, std)

    np.savez(channel_stats_path, variables=np.asarray(USED_VARIABLES), mean=mean.astype(np.float32), std=std.astype(np.float32))
    print("Saved channel stats:", channel_stats_path)
    print(pd.DataFrame({"variable": USED_VARIABLES, "mean": mean, "std": std}))
    return mean.astype(np.float32), std.astype(np.float32)


channel_mean, channel_std = build_or_load_channel_stats(
    cache_dir,
    valid_dates,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    sample_step=CHANNEL_STATS_SAMPLE_STEP,
    force=FORCE_REBUILD_CHANNEL_STATS,
)

dataset_daily = GridCacheDirectDataset(
    cache_dir=cache_dir,
    valid_dates=valid_dates,
    variables=USED_VARIABLES,
    history_len=HISTORY_LEN,
    horizon=HORIZON_DAYS,
    channel_mean=channel_mean,
    channel_std=channel_std,
)
train_ds_daily, val_ds_daily, test_ds_daily, target_frame_daily = split_direct_dataset_by_complete_target_window(dataset_daily)

if TRAIN_INDEX_STEP > 1:
    train_ds_daily = Subset(dataset_daily, list(train_ds_daily.indices)[::TRAIN_INDEX_STEP])
if VAL_INDEX_STEP > 1:
    val_ds_daily = Subset(dataset_daily, list(val_ds_daily.indices)[::VAL_INDEX_STEP])

train_loader = DataLoader(train_ds_daily, batch_size=UNET_BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds_daily, batch_size=UNET_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

print("dataset input channels:", dataset_daily.input_channels)
print("train/val/test:", len(train_ds_daily), len(val_ds_daily), len(test_ds_daily))



In [ ]:
# ============================================================
# LIGHTWEIGHT DIRECT U-NET MODEL
# ============================================================

class UNetConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups=min(8, out_channels), num_channels=out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups=min(8, out_channels), num_channels=out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)


class UNetDown(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv = UNetConvBlock(in_channels, out_channels)

    def forward(self, x):
        return self.conv(self.pool(x))


class UNetUp(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = UNetConvBlock(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = torch.nn.functional.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        return self.conv(torch.cat([x, skip], dim=1))


class SimpleDirectSICUNet(nn.Module):
    def __init__(self, in_channels, horizon=30, base_channels=16):
        super().__init__()
        c1 = int(base_channels)
        c2 = c1 * 2
        c3 = c1 * 4
        c4 = c1 * 8

        self.enc1 = UNetConvBlock(in_channels, c1)
        self.enc2 = UNetDown(c1, c2)
        self.enc3 = UNetDown(c2, c3)
        self.bottleneck = UNetDown(c3, c4)

        self.dec3 = UNetUp(c4, c3, c3)
        self.dec2 = UNetUp(c3, c2, c2)
        self.dec1 = UNetUp(c2, c1, c1)
        self.head = nn.Conv2d(c1, horizon, kernel_size=1)

    def forward(self, x, curr_sic):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        b = self.bottleneck(e3)

        d3 = self.dec3(b, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)

        delta = self.head(d1)
        pred = torch.clamp(curr_sic + delta, 0.0, 1.0)
        return {"delta": delta, "pred_full": pred}


def masked_sic_mse(pred, target):
    valid = torch.isfinite(target)
    if int(valid.sum().item()) == 0:
        return pred.sum() * 0.0
    return torch.mean((pred[valid] - target[valid]) ** 2)


def run_epoch(model, loader, device, optimizer=None, desc="epoch"):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    total_count = 0
    iterator = tqdm(loader, desc=desc, leave=False)
    for batch in iterator:
        x = batch["x"].to(device, non_blocking=True)
        curr_sic = batch["curr_sic"].to(device, non_blocking=True)
        y_full = batch["y_full"].to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)
        outputs = model(x, curr_sic)
        loss = masked_sic_mse(outputs["pred_full"], y_full)
        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        batch_size = int(x.size(0))
        total_loss += float(loss.item()) * batch_size
        total_count += batch_size
        iterator.set_postfix(loss=float(loss.item()))

    return total_loss / max(total_count, 1)


def fit_unet_model(model, train_loader, val_loader, device, save_path, epochs=20, patience=3, lr=1e-3, weight_decay=1e-5):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    best_val = math.inf
    wait = 0

    for epoch in range(1, int(epochs) + 1):
        train_loss = run_epoch(model, train_loader, device, optimizer=optimizer, desc=f"train {epoch}/{epochs}")
        val_loss = run_epoch(model, val_loader, device, optimizer=None, desc=f"val {epoch}/{epochs}")
        row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss}
        history.append(row)
        print(f"epoch {epoch:03d} | train={train_loss:.6f} | val={val_loss:.6f}")

        if val_loss < best_val:
            best_val = val_loss
            wait = 0
            torch.save(model.state_dict(), save_path)
            print("  saved best checkpoint:", save_path)
        else:
            wait += 1
            if wait >= patience:
                print("  early stopping after", wait, "non-improving epochs")
                break

    return pd.DataFrame(history)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model_unet = SimpleDirectSICUNet(
    in_channels=dataset_daily.input_channels,
    horizon=HORIZON_DAYS,
    base_channels=UNET_BASE_CHANNELS,
).to(device)

if UNET_LOAD_IF_EXISTS and unet_checkpoint_path.exists() and not UNET_FORCE_RETRAIN:
    print("Loading U-Net checkpoint:", unet_checkpoint_path)
    model_unet.load_state_dict(torch.load(unet_checkpoint_path, map_location=device))
    history_unet = None
else:
    history_unet = fit_unet_model(
        model_unet,
        train_loader,
        val_loader,
        device,
        save_path=unet_checkpoint_path,
        epochs=UNET_EPOCHS,
        patience=UNET_PATIENCE,
        lr=UNET_LR,
        weight_decay=UNET_WEIGHT_DECAY,
    )
    history_unet.to_csv(training_history_path, index=False)
    print("saved:", training_history_path)

if unet_checkpoint_path.exists():
    model_unet.load_state_dict(torch.load(unet_checkpoint_path, map_location=device))
    model_unet.eval()

